# HYCOM vs GLORYS — Fram Strait

Compares HYCOM TP2 output against GLORYS12 across Fram Strait using two approaches.  See [transects_transports.ipynb](transects_transports.ipynb) for a full walkthrough of the transport framework.

| Variable | Description |
|---|---|
| `fs_on_hycom` | Fram Strait transect resolved on the HYCOM C-grid |
| `tr_hycom` | HYCOM transport time series — full year 2020 |
| `sec0_hycom` | HYCOM hydrographic cross-section — 1 Jan 2020 |

In [1]:
import numpy as np
import xarray as xr
import xhycom
import matplotlib.pyplot as plt

In [2]:
GRID_PATH   = "/cluster/home/nlo043/NERSC-HYCOM-CICE/TP2a0.10/topo/regional.grid"
BATHY_PATH  = "/cluster/home/nlo043/NERSC-HYCOM-CICE/TP2a0.10/topo/depth_TP2a0.10_01"
DATA_PATH   = "/nird/datalake/NS9481K/shuang/TP2_output/expt_02.8/"
GLORYS_PATH = "/nird/datapeak/NS9481K/MERCATOR_DATA/PHY/2020/MERCATOR-PHY-24-2020-*.nc"

In [3]:
grid = xhycom.open_dataset(GRID_PATH)
bathy = xhycom.open_dataset(BATHY_PATH, grid=GRID_PATH)

In [4]:
fs = xhycom.Transect.named("fram_strait")
fs_on_hycom = fs.resolve(grid)

In [5]:
ds = xhycom.open_mfdataset(DATA_PATH + "archm.2020*", grid=GRID_PATH, chunks={"time": 1}, postprocess=True)

In [6]:
%time tr_hycom = xhycom.transport(ds, fs_on_hycom).load()
# Convert cftime → datetime64 for joint plotting with GLORYS
tr_hycom = tr_hycom.assign_coords(time=tr_hycom.indexes["time"].to_datetimeindex(time_unit="ns"))

CPU times: user 36.9 s, sys: 16.3 s, total: 53.2 s
Wall time: 1h 7min 35s


KeyboardInterrupt: 

In [ ]:
sec0_hycom = xhycom.section_data(ds, fs_on_hycom, variables=["temp", "salin"]).isel(time=0).compute()

---

## Comparison with GLORYS

In [ ]:
glorys_data = xr.open_mfdataset(GLORYS_PATH, chunks={"time": 1})

In [ ]:
glorys_data

Three approaches for comparing GLORYS against HYCOM:

| | **Method 1** — native grids | **Method 2** — HYCOM on GLORYS grid | **Method 3** — hybrid |
|---|---|---|---|
| **HYCOM** | native HYCOM grid | regridded horiz + vert | regridded vert only |
| **GLORYS** | native GLORYS grid | native GLORYS grid | regridded horiz only |
| **Shared horizontal grid** | — (different grids)  | GLORYS lon/lat | HYCOM curvilinear (y, x) |
| **Shared vertical grid** | — (different grids) | GLORYS depth | GLORYS depth |
| **Sections** | ✔ | ✔ | ✔ |
| **Section diff plots** | — (different grids) | ✔ | ✔ |
| **Transport** | ✔ | ✔ | ✔ |
| **Transport diff plots** | ✔ | ✔ | ✔ |
| **Regrid cost** | none | horiz + vert (HYCOM) | horiz (GLORYS) + vert (HYCOM) |


### Helper functions

In [ ]:
def plot_hycom_vs_glorys(hycom_sec, glorys_sec):
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=True, sharex=True)
    
    depth_max = 3000
    t_kw = {"cmap": "RdYlBu_r", "vmin": -2, "vmax": 3.5}
    s_kw = {"cmap": "viridis", "vmin": 33, "vmax": 35}
    
    xhycom.section_plot(hycom_sec,   "temp",   ax=axes[0, 0], title="HYCOM temperature (°C)",  depth_max=depth_max, flip_x=True, **t_kw)
    xhycom.section_plot(glorys_sec, "thetao", ax=axes[0, 1], title="GLORYS temperature(°C)", depth_var="depth", depth_max=depth_max, flip_x=True, **t_kw)
    xhycom.section_plot(hycom_sec,   "salin",   ax=axes[1, 0], title="HYCOM salinity (psu)",  depth_max=depth_max, flip_x=True, **s_kw)
    xhycom.section_plot(glorys_sec, "so", ax=axes[1, 1], title="GLORYS salinity (psu)", depth_var="depth", depth_max=depth_max, **s_kw)
    
    fig.suptitle(f"HYCOM versus GLORYS in the FRAM Strait on {str(ds.time.values[0])[:10]}", fontsize=13)
    plt.tight_layout()

    return fig

In [ ]:
def plot_transport_comparison(tr_hycom, others):
    """Plot HYCOM transport against one or more other datasets.

    Parameters
    ----------
    tr_hycom : xr.Dataset
        HYCOM native transport.
    others : dict[str, xr.Dataset]
        Mapping of label → transport Dataset to overlay.
    """
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    pairs = [
        (axs[0, 0], "volume", "Volume transport (Sv)"),
        (axs[0, 1], "heat",   "Heat transport (TW)"),
        (axs[1, 0], "salt",   "Salt transport (kg s\u207b\u00b9)"),
        (axs[1, 1], "fw",     "Freshwater transport (Sv)"),
    ]
    for ax, var, title in pairs:
        tr_hycom[var].plot(ax=ax, label="HYCOM")
        for label, tr in others.items():
            if var in tr:
                tr[var].plot(ax=ax, label=label)
        ax.set_title(title)
        ax.grid()
        ax.legend()
    plt.tight_layout()
    return fig


### Method 1 — native grids

Pass `lat_var` and `lon_var` so `resolve()` builds its KD-tree over GLORYS coordinates — no interpolation, fields extracted at the nearest GLORYS T-cells.

In [ ]:
%time fs_on_glorys = fs.resolve(glorys_data, lat_var="latitude", lon_var="longitude")

The two sections sit on different horizontal grids (57 HYCOM T-cells vs ~115 GLORYS cells at 0.083°), so the comparison is visual only — pixel-level subtraction is not valid here. Use Method 2 for difference plots.

In [ ]:
%time glorys_sec0 = xhycom.section_data(glorys_data, fs_on_glorys, variables=["thetao", "so"]).isel(time=0).compute()

In [ ]:
fig = plot_hycom_vs_glorys(sec0_hycom, glorys_sec0)

For non-HYCOM data, pass `z_dim`, `t_var`, and `s_var`; `transport()` integrates velocity over depth-coordinate spacing rather than hybrid layer thicknesses. Face-normal velocities are sampled at the nearest GLORYS grid cell rather than at exact C-grid face crossings (as in Methods 2 and 3).

In [ ]:
%time tr_glorys = xhycom.transport(glorys_data, fs_on_glorys, z_dim="depth", u_var="uo", v_var="vo", t_var="thetao", s_var="so").compute()

In [ ]:
tr_glorys

In [ ]:
fig = plot_transport_comparison(tr_hycom, {"Method 1 — GLORYS native": tr_glorys})

Transport *differences* are valid even without regridding: transport is an integrated scalar (Sv, TW, …) with no remaining grid dependence once summed across all section faces and depth levels.

In [ ]:
(tr_hycom.volume - tr_glorys.volume).plot()

### Method 2 — HYCOM on GLORYS grid

`xhycom.regrid()` remaps HYCOM's curvilinear, hybrid-layer output onto the GLORYS lon/lat/depth grid in one call (conservative horizontal + conservative vertical by default). Because both datasets end up on the same regular grid, you get diff plots for free and can call `transport()` on the regridded HYCOM the same way as on native GLORYS.


In [ ]:
%time ds_on_glorys = xhycom.regrid(ds.isel(time=0), target=glorys_data, grid=GRID_PATH, variables=["temp", "salin", "u-vel.", "v-vel."]).compute()

In [ ]:
ds_on_glorys

In [ ]:
%time sec0_hycom_m2 = xhycom.section_data(ds_on_glorys, fs_on_glorys, lat_var="latitude", lon_var="longitude", variables=["temp", "salin"]).compute()

In [ ]:
fig = plot_hycom_vs_glorys(sec0_hycom_m2, glorys_sec0)

In [ ]:
%time tr_hycom_m2 = xhycom.transport(ds_on_glorys, fs_on_glorys, z_dim="depth", u_var="u-vel.", v_var="v-vel.", t_var="temp", s_var="salin").load()

In [ ]:
tr_hycom_m2

### Method 3 — GLORYS on HYCOM grid / HYCOM on GLORYS depths

Two targeted regrids onto a hybrid grid (HYCOM curvilinear × GLORYS depth levels):

- GLORYS is regridded **horizontally** onto the HYCOM `(y, x)` grid (`regrid_to_hycom`); depth levels stay as GLORYS standard levels.
- HYCOM is regridded **vertically** onto GLORYS depth levels (`regrid_vertical`); the curvilinear horizontal grid is unchanged.

Both datasets end up on the same `(y, x, depth)` grid, so `fs_m3` (a T-point resolve on the HYCOM grid) covers both and cell-by-cell differences are valid.

In [ ]:
%time glorys_on_hycom = xhycom.regrid_to_hycom(glorys_data.isel(time=0), GRID_PATH, like=ds).compute()

In [ ]:
%time ds_on_glorys_depth = xhycom.regrid_vertical(ds.isel(time=0), glorys_data.depth.values).compute()

In [ ]:
# Both share the same HYCOM (y, x) grid — one T-point resolve covers both
%time fs_m3 = fs.resolve(glorys_on_hycom, lat_var="lat", lon_var="lon")

In [ ]:
sec0_hycom_m3  = xhycom.section_data(ds_on_glorys_depth, fs_m3, variables=["temp",   "salin"], z_dim="depth").compute()
sec0_glorys_m3 = xhycom.section_data(glorys_on_hycom,    fs_m3, variables=["thetao",  "so"  ], z_dim="depth").compute()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=True, sharex=True)
depth_max = 3000
t_kw = {"cmap": "RdYlBu_r", "vmin": -2, "vmax": 3.5}
s_kw = {"cmap": "viridis", "vmin": 33, "vmax": 35}

xhycom.section_plot(sec0_hycom_m3,  "temp",   ax=axes[0, 0], title="HYCOM temperature (°C)",  depth_max=depth_max, flip_x=True, **t_kw)
xhycom.section_plot(sec0_glorys_m3, "thetao", ax=axes[0, 1], title="GLORYS temperature (°C)", depth_max=depth_max, flip_x=True, **t_kw)
xhycom.section_plot(sec0_hycom_m3,  "salin",  ax=axes[1, 0], title="HYCOM salinity (psu)",    depth_max=depth_max, flip_x=True, **s_kw)
xhycom.section_plot(sec0_glorys_m3, "so",     ax=axes[1, 1], title="GLORYS salinity (psu)",   depth_max=depth_max, **s_kw)
fig.suptitle(f"Method 3 — HYCOM (GLORYS depths) vs GLORYS (HYCOM grid) — {str(ds.time.values[0])[:10]}", fontsize=13)
plt.tight_layout()

In [ ]:
%time tr_hycom_m3  = xhycom.transport(ds_on_glorys_depth, fs_m3, z_dim="depth", u_var="u-vel.", v_var="v-vel.", t_var="temp",   s_var="salin" ).compute()
%time tr_glorys_m3 = xhycom.transport(glorys_on_hycom,    fs_m3, z_dim="depth", u_var="uo",     v_var="vo",     t_var="thetao", s_var="so"    ).compute()

### Comparing all transports

Side-by-side of all transport estimates: HYCOM native, GLORYS native (Method 1), HYCOM on GLORYS grid (Method 2), and Method 3 (HYCOM on GLORYS depths / GLORYS on HYCOM grid).

In [ ]:
fig = plot_transport_comparison(tr_hycom, {
    "Method 1 — GLORYS native":          tr_glorys,
    "Method 2 — HYCOM on GLORYS":        tr_hycom_m2,
    "Method 3 — HYCOM (GLORYS depths)":  tr_hycom_m3,
    "Method 3 — GLORYS (HYCOM grid)":    tr_glorys_m3,
})